#Problem 1 — Deadlock Detection and Recovery
#Concept: Circular lock dependency

In [ ]:
import asyncio
import time
import nest_asyncio
nest_asyncio.apply()

from distributed import sync
lockA=asyncio.Lock()
lockB=asyncio.Lock()

async def task1():
    async with lockA:
        print("Task 1 acquired lock A ") 
        await asyncio.sleep(1)  # Simulate some work
        print("Task 1 waiting for lock B")
        async with lockB:
            print("Task 1 acquired lock B") 
            await asyncio.sleep(1)  # Simulate some work
            print("Task 1 released lock B")


async def task2():
    async with lockB:
        print("Task 2 acquired lock B") 
        await asyncio.sleep(1)  # Simulate some work
        print("Task 2 waiting for lock A")
        async with lockA:
            print("Task 2 acquired lock A") 
            await asyncio.sleep(1)  # Simulate some work
            print("Task 2 released lock A")

async def main():
    start_time = time.time() 
    tasks = await asyncio.gather(task1(), task2())
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time} seconds")

if __name__ == "__main__":
    asyncio.run(main())


Task 1 acquired lock A 
Task 2 acquired lock B
Task 1 waiting for lock B
Task 2 waiting for lock A


In [1]:
# recovery from deadlock using sequential acquisition of locks
import asyncio
import time
import nest_asyncio
nest_asyncio.apply()

from distributed import sync
lockA=asyncio.Lock()
lockB=asyncio.Lock()

async def task1():
    async with lockA:
        print("Task 1 acquired lock A ") 
        await asyncio.sleep(1)  # Simulate some work
        print("Task 1 waiting for lock B")
        async with lockB:
            print("Task 1 acquired lock B") 
            await asyncio.sleep(1)  # Simulate some work
            print("Task 1 released lock B")


async def task2():
    async with lockA:
        print("Task 2 acquired lock A") 
        await asyncio.sleep(1)  # Simulate some work
        print("Task 2 waiting for lock B")
        async with lockB:
            print("Task 2 acquired lock B") 
            await asyncio.sleep(1)  # Simulate some work
            print("Task 2 released lock B")

async def main():
    start_time = time.time() 
    tasks = await asyncio.gather(task1(), task2())
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time} seconds")

if __name__ == "__main__":
    asyncio.run(main())


Task 1 acquired lock A 
Task 1 waiting for lock B
Task 1 acquired lock B
Task 1 released lock B
Task 2 acquired lock A
Task 2 waiting for lock B
Task 2 acquired lock B
Task 2 released lock B
Total execution time: 4.032123804092407 seconds


In [2]:
# Recovery from deadlock using wait_for with timeout
import asyncio
import time
import nest_asyncio
nest_asyncio.apply()

from distributed import sync
lockA=asyncio.Lock()
lockB=asyncio.Lock()

async def task1():
    async with lockA:
        print("Task 1 acquired lock A") 
        await asyncio.sleep(1)  # Simulate some work
        print("Task 1 waiting for lock B")
        try:
            await asyncio.wait_for(lockB.acquire(), timeout=2)
            try:
                print("Task 1 acquired lock B") 
                await asyncio.sleep(1)  # Simulate some work
                print("Task 1 released lock B")
            finally:
                lockB.release()
        except asyncio.TimeoutError:
            print("Task 1 timed out waiting for lock B")


async def task2():
    async with lockB:
        print("Task 2 acquired lock B")
        await asyncio.sleep(1)  # Simulate some work
        print("Task 2 waiting for lock A")
        try:
            await asyncio.wait_for(lockA.acquire(),timeout=2)        
            try:
                print("Task2 has acquired Lock A and is doing some work")
                await asyncio.sleep(1)
                print("Task 2 released lock A")   
            finally:
                lockA.release()    
        except asyncio.TimeoutError:
            print("Task 2 timed out waiting for lock A")

async def main():
    start_time = time.time() 
    tasks = await asyncio.gather(task1(), task2())
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time} seconds")

if __name__ == "__main__":
    asyncio.run(main())                

Task 1 acquired lock A
Task 2 acquired lock B
Task 1 waiting for lock B
Task 2 waiting for lock A
Task 1 timed out waiting for lock B
Task 2 timed out waiting for lock A
Total execution time: 3.027411937713623 seconds


In [7]:
# Run Semaphore example

import asyncio
import random
import time
import nest_asyncio
nest_asyncio.apply()
semaphore=asyncio.Semaphore(2)  # Limit to 2 concurrent workers

async def worker(worker_id:int):
      start_time = time.time()
      async with semaphore:
          print (f"Worker {worker_id} is starting work.")
          await asyncio.sleep(1)  # Simulate work
          print (f"Worker {worker_id} has finished work.")
      end_time = time.time()
      print(f"Worker {worker_id} execution time: {end_time - start_time:.2f} seconds")


async def main():
     start_time = time.time()
     tasks=[worker(worker_id) for worker_id in range(1,6)]
     await asyncio.gather(*tasks)
     end_time = time.time()
     print(f"Total execution time: {end_time - start_time:.2f} seconds")

if __name__ == "__main__":
    asyncio.run(main())


Worker 1 is starting work.
Worker 2 is starting work.
Worker 1 has finished work.
Worker 1 execution time: 1.01 seconds
Worker 2 has finished work.
Worker 2 execution time: 1.01 seconds
Worker 3 is starting work.
Worker 4 is starting work.
Worker 3 has finished work.
Worker 3 execution time: 2.02 seconds
Worker 4 has finished work.
Worker 4 execution time: 2.02 seconds
Worker 5 is starting work.
Worker 5 has finished work.
Worker 5 execution time: 3.03 seconds
Total execution time: 3.03 seconds


In [13]:
#  create_task() vs gather() in asyncio
import asyncio
import random
import time
import nest_asyncio
nest_asyncio.apply()


async def say_hello(name:str,delay:int):
    start_time = time.time()
    await asyncio.sleep(delay)
    print(f"Hello, {name} after {delay} seconds!")
    end_time = time.time()
    print(f"{name} Actual execution time: {end_time - start_time:.2f} seconds")


async def main():
    start_time = time.time()
    
    print("Using asyncio.gather()")
    await asyncio.gather(
        say_hello("Alice", 0.1),
        say_hello("Bob", 0.2),
        say_hello("Charlie", 0.3)
    )
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time:.2f} seconds")
    

    print("\nUsing asyncio.create_task()")  
    start_time = time.time()
    asyncio.create_task(say_hello("X", 0.1))
    asyncio.create_task(say_hello("Y", 0.2))
    asyncio.create_task(say_hello("Z", 0.3))
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time:.2f} seconds")
    await asyncio.sleep(1)  # Wait for all tasks to complete

async def main_no_sleep():
    print("No-sleep version using asyncio.create_task()")
    start_time = time.time()
    asyncio.create_task(say_hello("X", 0.1))
    asyncio.create_task(say_hello("Y", 0.2))
    asyncio.create_task(say_hello("Z", 0.3))
      # Wait for all tasks to complete
    end_time = time.time()
    print(f"Total execution time: {end_time - start_time:.2f} seconds")


if __name__ == "__main__":
    #asyncio.run(main())
    asyncio.run(main_no_sleep())
    




No-sleep version using asyncio.create_task()
Total execution time: 0.00 seconds


Hello, X after 0.1 seconds!
X Actual execution time: 0.10 seconds
Hello, Y after 0.2 seconds!
Y Actual execution time: 0.21 seconds
Hello, Z after 0.3 seconds!
Z Actual execution time: 0.31 seconds
